<h1>Chapter 2 - Generation Models</h1>
<i>Choosing the generation model for your RAG system.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch02_generation/generation.ipynb)

---

This notebook is for Chapter 2 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


In [1]:
%pip install openai==2.21.0 anthropic==0.18.1 python-dotenv==1.0.0 httpx==0.27.0

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 9.7 MB/s  0:00:00
   ---------------------------------------- 0.0/848.1 kB ? eta -:--:--
   ---------------------------------------- 848.1/848.1 kB 15.1 MB/s  0:00:00

  Attempting uninstall: httpx

    Found existing installation: httpx 0.28.1

    Uninstalling httpx-0.28.1:

      Successfully uninstalled httpx-0.28.1

   ---------------------------------------- 0/3 [httpx]
   ---------------------------------------- 0/3 [httpx]
   ---------------------------------------- 0/3 [httpx]
   ---------------------------------------- 0/3 [httpx]
   ---------------------------------------- 0/3 [httpx]
   ---------------------------------------- 0/3 [httpx]
   ---------------------------------------- 0/3 [httpx]
  Attempting uninstall: openai
   ---------------------------------------- 0/3 [httpx]
    Found existing installation: openai 1.109.1
   -----------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.3.21 requires openai<2.0.0,>=1.68.2, but you have openai 2.21.0 which is incompatible.
mcp 1.26.0 requires httpx>=0.27.1, but you have httpx 0.27.0 which is incompatible.
openai-agents 0.9.2 requires pydantic<3,>=2.12.3, but you have pydantic 2.11.7 which is incompatible.


### Load secrets

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [2]:
import os
import sys
from dotenv import load_dotenv

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

### Load sample files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [3]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

⚠ Running locally. Using ../datasets/ directory


## 1. OpenAI Chat Completions

In [4]:
from openai import OpenAI

def ask_with_context(context, question):
    client = OpenAI()

    messages = [
        {
            "role": "system",
            "content": "Answer based only on the provided context."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion:\n{question}"
        },
    ]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    return response.choices[0].message.content


# Usage
context = "RAG stands for Retrieval-Augmented Generation."
question = "What does RAG stand for?"
answer = ask_with_context(context, question)
print(answer)

RAG stands for Retrieval-Augmented Generation.


## 2. OpenAI Whisper Speech-to-Text

In [5]:
import os
os.listdir()

['generation.ipynb']

In [6]:
from openai import OpenAI
import os

client = OpenAI()

# Determine the correct path based on environment
if IN_COLAB:
    audio_path = "/content/datasets/audio_files/harvard.wav"
else:
    audio_path = "../datasets/audio_files/harvard.wav"

with open(audio_path, "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
    )

print(transcript.text)

The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun.


## 3. Anthropic Claude Example

In [7]:
client.models.list()

SyncPage[Model](data=[Model(id='gpt-4-0613', created=1686588896, object='model', owned_by='openai'), Model(id='gpt-4', created=1687882411, object='model', owned_by='openai'), Model(id='gpt-3.5-turbo', created=1677610602, object='model', owned_by='openai'), Model(id='gpt-5.4', created=1772691852, object='model', owned_by='system'), Model(id='gpt-5.3-chat-latest', created=1772236571, object='model', owned_by='system'), Model(id='gpt-5.4-2026-03-05', created=1772654062, object='model', owned_by='system'), Model(id='gpt-5.4-pro', created=1772659601, object='model', owned_by='system'), Model(id='gpt-5.4-pro-2026-03-05', created=1772659657, object='model', owned_by='system'), Model(id='davinci-002', created=1692634301, object='model', owned_by='system'), Model(id='babbage-002', created=1692634615, object='model', owned_by='system'), Model(id='gpt-3.5-turbo-instruct', created=1692901427, object='model', owned_by='system'), Model(id='gpt-3.5-turbo-instruct-0914', created=1694122472, object='mo

In [8]:
from anthropic import Anthropic
import os

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=200,
    messages=[
        {
            "role": "user",
            "content": "Explain how vector databases work in "
                       "simple terms.",
        }
    ],
)

print(response.content[0].text)

# Vector Databases - Simple Explanation

## The Basic Idea

A **vector database** stores information as lists of numbers (vectors) instead of traditional text. It's like giving every piece of data a unique "numerical fingerprint" that captures its meaning.

## How It Works

**1. Converting to Vectors**
- Text, images, or other data get transformed into vectors (arrays of numbers)
- Similar things get similar numbers
- Example: "cat" and "kitten" would have very similar vectors

**2. Storing**
- These vectors are stored in a way that makes finding similar items fast
- Think of it like organizing items by their characteristics, not just alphabetically

**3. Searching**
- Instead of exact keyword matching, you search by *meaning*
- The database finds vectors that are mathematically "close" to your query
- Returns the most similar items, even if they use different words


## 4. Gemini API Example using the OpenAI SDK

In [9]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

client.models.list()

SyncPage[Model](data=[Model(id='models/gemini-2.5-flash', created=None, object='model', owned_by='google', display_name='Gemini 2.5 Flash'), Model(id='models/gemini-2.5-pro', created=None, object='model', owned_by='google', display_name='Gemini 2.5 Pro'), Model(id='models/gemini-2.0-flash', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash'), Model(id='models/gemini-2.0-flash-001', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash 001'), Model(id='models/gemini-2.0-flash-lite-001', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash-Lite 001'), Model(id='models/gemini-2.0-flash-lite', created=None, object='model', owned_by='google', display_name='Gemini 2.0 Flash-Lite'), Model(id='models/gemini-2.5-flash-preview-tts', created=None, object='model', owned_by='google', display_name='Gemini 2.5 Flash Preview TTS'), Model(id='models/gemini-2.5-pro-preview-tts', created=None, object='model', owned_by='goo

In [10]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

resp = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
)

print(resp.choices[0].message.content)

The capital of France is **Paris**.


## 5. Deploy local LLMs using Ollama

### Setting up Ollama in Colab

Since Colab runs in a cloud environment, you need to install and run Ollama directly within the Colab instance. The following steps will set up Ollama and pull the required models.

In [11]:
import subprocess
import os

# Install zstd dependency
!sudo apt-get update && sudo apt-get install -y zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
os.environ['OLLAMA_HOST'] = '0.0.0.0'
subprocess.Popen(["ollama", "serve"])

print("Ollama server is starting...")

Sudo is disabled by your organization's policy.


Ollama server is starting...


'sh' is not recognized as an internal or external command,
operable program or batch file.


In [12]:
# Give Ollama some time to ensure it's fully started (adjust if needed)
import time
time.sleep(10)

print("Pulling models...")
!ollama pull qwen3:4b
!ollama pull llama2
!ollama pull mistral
!ollama pull codellama

print("Models pulled successfully. You can now re-run the Ollama examples.")

Pulling models...


pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest â § pulling manifest â ‡ pulling manifest 
pulling 3e4cb1417446: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 2.5 GB                         
pulling 2d54db2b9bb2: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 1.5 KB                         
pulling d18a5cc71b84: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  11 KB                         
pulling cff3f395ef37: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  120 B                         
pulling e18a783aae55: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  487 B                         
verifying sha256 digest 
writing manifest 
success 


pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest â § pulling manifest â ‡ pulling manifest â � pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest â § pulling manifest 
pulling 8934d96d3f08:   0% â–•                  â–�  92 KB/3.8 GB                  pulling manifest 
pulling 8934d96d3f08:   0% â–•                  â–� 1.7 MB/3.8 GB                  pulling manifest 
pulling 8934d96d3f08:   0% â–•                  â–� 6.7 MB/3.8 GB                  pulling manifest 
pulling 8934d96d3f08:   0% â–•                  â–�  10 MB/3.8 GB                  pulling manifest 
pulling 8934d96d3f08:   0% â–•                  â–�  16 MB/3.8 GB                  pulling manifest 
pulling 8934d96d3f08:   1% â–•                  â–�  23 MB/3.8 GB                  pulling manifest

pulling manifest â ‹ pulling manifest â ™ pulling manifest â ¹ pulling manifest â ¸ pulling manifest â ¼ pulling manifest â ´ pulling manifest â ¦ pulling manifest â § pulling manifest â ‡ pulling manifest â � pulling manifest â ‹ pulling manifest â ™ pulling manifest 
pulling f5074b1221da:   0% â–•                  â–� 1.1 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   0% â–•                  â–� 3.5 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   0% â–•                  â–�  10 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   0% â–•                  â–�  18 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   1% â–•                  â–�  24 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   1% â–•                  â–�  28 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   1% â–•                  â–�  34 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:  


verifying sha256 digest â ‡ pulling manifest 
pulling f5074b1221da: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 4.4 GB                         
pulling 43070e2d4e53: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  11 KB                         
pulling 1ff5b64b61b9: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  799 B                         
pulling ed11eda7790d: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�   30 B                         
pulling 1064e17101bd: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  487 B                         
verifying sha256 digest â � pulling manifest 
pulling f5074b1221da: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 4.4 GB                         
pulling 43070e2d4e53: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  11 KB                         
pulling 1ff5b64b61b9: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ

...

verifying sha256 digest â ¹ pulling manifest 
pulling 3a43f93b78ec: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 3.8 GB                         
pulling 8c17c2ebb0ea: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 7.0 KB                         
pulling 590d74a5569b: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 4.8 KB                         
pulling 2e0493f67d0c: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�   59 B                         
pulling 7f6a57943a88: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  120 B                         
pulling 316526ac7323: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  529 B                         
verifying sha256 digest â ¸ pulling manifest 
pulling 3a43f93b78ec: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 3.8 GB                         
pulling 8c17c2ebb0ea: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–

pulling 8c17c2ebb0ea: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 7.0 KB                         
pulling 590d74a5569b: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 4.8 KB                         
pulling 2e0493f67d0c: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�   59 B                         
pulling 7f6a57943a88: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  120 B                         
pulling 316526ac7323: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  529 B                         
verifying sha256 digest â ¼ pulling manifest 
pulling 3a43f93b78ec: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 3.8 GB                         
pulling 8c17c2ebb0ea: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 7.0 KB                         
pulling 590d74a5569b: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 4.8 KB                    


pulling 3a43f93b78ec: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 3.8 GB                         
pulling 8c17c2ebb0ea: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 7.0 KB                         
pulling 590d74a5569b: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 4.8 KB                         
pulling 2e0493f67d0c: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�   59 B                         
pulling 7f6a57943a88: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  120 B                         
pulling 316526ac7323: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  529 B                         
verifying sha256 digest â ¦ pulling manifest 
pulling 3a43f93b78ec: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 3.8 GB                         
pulling 8c17c2ebb0ea: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–� 7.0 KB                   

Models pulled successfully. You can now re-run the Ollama examples.



pulling 7f6a57943a88: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  120 B                         
pulling 316526ac7323: 100% â–•â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–�  529 B                         
verifying sha256 digest 
writing manifest 
success 


In [13]:
from openai import OpenAI

# Point the client to your local Ollama server
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # Ollama does not require a real key,
                       # but the SDK expects one
)

response = client.chat.completions.create(
    model="qwen3:4b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "What is retrieval augmented generation?"
        },
    ],
)

print(response.choices[0].message.content)

**Retrieval Augmented Generation (RAG)** is an AI technique that combines **retrieval of relevant information** from an external knowledge source with **generative language modeling** to produce more accurate, context-aware, and up-to-date responses. It’s designed to overcome the limitations of standalone language models (like GPT) that may lack recent data, domain-specific knowledge, or domain expertise.

---

### 🔍 How It Works (Step-by-Step)
1. **User Query**: A user asks a question (e.g., *"What’s the current population of France?"*).
2. **Retrieval**: 
   - The system searches a **knowledge base** (e.g., internal documents, databases, or the web) using semantic similarity.
   - It retrieves the most relevant documents or fragments (e.g., recent population stats from a reliable source).
3. **Generation**:
   - The retrieved information is fed into a language model (e.g., a fine-tuned LLM).
   - The model generates a **precise, context-aware response** using the retrieved data.

> 💡

In [14]:
from openai import OpenAI

models = ["llama2", "mistral", "codellama"]
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

for model in models:
    print(f"\n--- Testing {model} ---")
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": "Explain RAG in one sentence."}
        ]
    )
    print(response.choices[0].message.content)


--- Testing llama2 ---



RAG (Red, Amber, Green) is a visual management tool used to track and monitor progress, status, and performance in a simple and intuitive way by assigning a color code (red, amber, or green) to each category of data, with red indicating a problem or issue that needs attention, amber indicating a developing problem or potential issue, and green indicating all is well or progress is being made.

--- Testing mistral ---


 RAG refers to a color-coded rating system used to communicate the status or priority level of tasks or projects, where R represents Red for urgent high-priority items, A for Amber for medium-priority items, and G for Green for low-priority items.

--- Testing codellama ---



RAG, or Red, Amber, and Green, is a framework for evaluating the quality of a project or a product and providing a clear understanding of its level of completeness or readiness.


## 6. Pydantic Structured Output

In [15]:
from openai import OpenAI
from pydantic import BaseModel
from datetime import date
from typing import List

class LineItem(BaseModel):
    description: str
    quantity: int
    total: float

class Invoice(BaseModel):
    invoice_number: str
    invoice_date: date
    supplier: str
    items: List[LineItem]
    total_due: float

client = OpenAI()

completion = client.chat.completions.parse(
    model="gpt-5",
    messages=[
        {"role": "system", "content": "Extract the invoice data from the provided context."},
        {"role": "user", "content": "Invoice #12345, Date: 2024-01-15, Supplier: Tech Corp. Item: Laptop, Qty: 2, Total: $2000. Item: Mouse, Qty: 5, Total: $100. Total Due: $2100"}
    ],
    response_format=Invoice,
)

invoice = completion.choices[0].message.parsed
print(invoice)

invoice_number='12345' invoice_date=datetime.date(2024, 1, 15) supplier='Tech Corp' items=[LineItem(description='Laptop', quantity=2, total=2000.0), LineItem(description='Mouse', quantity=5, total=100.0)] total_due=2100.0
